# Module 06 — Notebook 2 Solutions: argparse and logging

In [ ]:
import sys
sys.path.insert(0, "../../../")
from src.checks import check_equal, check_type, check_approx, check_contains
import argparse
import logging
from pathlib import Path
SCRIPTS_DIR = Path("scripts")
SCRIPTS_DIR.mkdir(exist_ok=True)

## Exercise 1 Solution

In [ ]:
eval_parser = argparse.ArgumentParser(description="Evaluation runner")
eval_parser.add_argument("--input",     type=Path, required=True)
eval_parser.add_argument("--output",    type=Path, default=Path("output/results.json"))
eval_parser.add_argument("--threshold", type=float, default=0.8)
eval_parser.add_argument("--model",     type=str,  default=None)

test_args = eval_parser.parse_args(["--input", "data.json", "--threshold", "0.7"])

parsed_threshold = float(test_args.threshold)
parsed_output    = test_args.output   # already a Path because type=Path

In [ ]:
check_type(eval_parser, argparse.ArgumentParser, "eval_parser is an ArgumentParser")
check_type(parsed_threshold, float, "parsed_threshold is a float")
check_approx(parsed_threshold, 0.7, 1e-6, "parsed_threshold is 0.7")
check_type(parsed_output, Path, "parsed_output is a Path")
check_equal(str(parsed_output), "output/results.json", "parsed_output has correct default")

## Exercise 2 Solution

In [ ]:
pipeline_logger = logging.getLogger("eval_pipeline")
pipeline_logger.setLevel(logging.WARNING)
log_level = pipeline_logger.level   # 30 (logging.WARNING == 30)

In [ ]:
check_type(pipeline_logger, logging.Logger, "pipeline_logger is a Logger")
check_equal(int(log_level), 30, "log level is 30 (WARNING)")

## Exercise 3 Solution

In [ ]:
%%writefile scripts/score_filter.py
import argparse
import logging
import pandas as pd
from pathlib import Path


def build_parser():
    parser = argparse.ArgumentParser(description="Filter evaluation scores")
    parser.add_argument("--input",     type=Path, required=True)
    parser.add_argument("--min-score", type=float, default=0.8, dest="min_score")
    return parser


def main(args):
    logging.basicConfig(level=logging.INFO, format="%(levelname)s  %(message)s")
    logger = logging.getLogger(__name__)

    df = pd.read_csv(args.input)
    logger.info(f"Loaded {len(df)} rows from {args.input}")

    passing = df[df["score"] >= args.min_score]
    logger.info(f"Passing rows (score >= {args.min_score}): {len(passing)}")

    for _, row in passing.iterrows():
        logger.info(f"  {row['model']:15s} | {row['task']:25s} | {row['score']:.2f}")


if __name__ == "__main__":
    main(build_parser().parse_args())

In [ ]:
!python scripts/score_filter.py --input ../../../data/synthetic/evaluation_results.csv --min-score 0.9

In [ ]:
script_path = Path("scripts/score_filter.py")
check_equal(script_path.exists(), True, "score_filter.py exists")
source = script_path.read_text()
check_contains(source, "argparse",  "script imports argparse")
check_contains(source, "logging",   "script imports logging")
check_contains(source, "__name__",  "script has __name__ guard")
check_contains(source, "min_score", "script has min_score argument")